In [127]:
import pandas as pd
import numpy as np

In [128]:
cc_calls = pd.read_csv('../../data/raw/cc_calls.csv', low_memory=False)
cc_calls.columns = cc_calls.columns.str.lower().str.replace(' ', '_')
cc_calls.shape

(32882, 33)

Dropping duplicate rows

In [129]:
cc_calls.duplicated().sum()

np.int64(93)

In [130]:
cc_calls = cc_calls.drop_duplicates()
cc_calls.shape

(32789, 33)

Dropping rows with null co_ref

In [131]:
cc_calls['co_ref'].isnull().sum()

np.int64(1153)

In [132]:
cc_calls = cc_calls.dropna(subset=['co_ref'])
cc_calls.shape

(31636, 33)

Standardizing values in direction column

In [133]:
cc_calls['direction'].value_counts()

direction
OUT_BOUND    24292
IN_BOUND      7344
Name: count, dtype: int64

In [134]:
cc_calls['direction'] = cc_calls['direction'].replace({
    'OUT_BOUND': 'Outbound',
    'IN_BOUND': 'Inbound'
})
cc_calls['direction'].value_counts()

direction
Outbound    24292
Inbound      7344
Name: count, dtype: int64

Converting proper datatype of dates

In [135]:
cc_calls['call_date'].head()

0    08-05-2025
1    25-11-2024
2    23-10-2024
3    13-01-2025
4    19-03-2025
Name: call_date, dtype: str

In [136]:
cc_calls['call_date'] = pd.to_datetime(cc_calls['call_date'], format='%d-%m-%Y')
cc_calls['call_date'].head()

0   2025-05-08
1   2024-11-25
2   2024-10-23
3   2025-01-13
4   2025-03-19
Name: call_date, dtype: datetime64[us]

Handling score columns

In [137]:
score_columns = [
    'cc_contractor_sentiment_start_score', 
    'cc_contractor_sentiment_end_score', 
    'cc_contractor_sentiment_overall_score', 
    'cc_contractor_sentiment_issues_score'
]
cc_calls[score_columns].dtypes

cc_contractor_sentiment_start_score      str
cc_contractor_sentiment_end_score        str
cc_contractor_sentiment_overall_score    str
cc_contractor_sentiment_issues_score     str
dtype: object

In [138]:
for col in score_columns:
    cc_calls[col] = pd.to_numeric(cc_calls[col], errors='coerce')
cc_calls[score_columns].dtypes

cc_contractor_sentiment_start_score      float64
cc_contractor_sentiment_end_score        float64
cc_contractor_sentiment_overall_score    float64
cc_contractor_sentiment_issues_score     float64
dtype: object

Dropping irrelevant columns<br>
1- analysed_call -> has only one unique value<br>
2- call_year -> can be derived from call_date<br>
3- contact_id -> is unique identifier so wont be used for prediction

In [139]:
cc_calls['analysed_call'].unique()

array([1])

In [140]:
cc_calls['call_year'].unique()

array([2025, 2024, 2026])

In [141]:
cc_calls['contact_id'].nunique()

494

In [142]:
cc_calls = cc_calls.drop(columns=['analysed_call', 'call_year', 'contact_id'])
cc_calls.shape

(31636, 30)

Handling null values in cc_calls

In [143]:
cc_calls.isnull().sum()[cc_calls.isnull().sum() > 0]

cc_care_package                               136
cc_care_package_discussed                     136
cc_urgency_getting_on_site                    136
cc_external_consultant                        136
cc_agent_cross_sell_attempt                   136
cc_customer_issues_concerns                   136
cc_business_struggles_financial_hardship      136
cc_call_initiated_by                          136
cc_questionnaire_completion                    32
cc_chasing_response                            32
cc_issues_within_questionnaire                443
cc_login_issues                                33
cc_platform_issues                             33
cc_dissatisfaction_time_to_complete            33
cc_process_complexity_concerns                 38
cc_questions_harder_than_expected              37
cc_dissatisfaction_support                     36
cc_contractor_sentiment                        95
cc_contractor_sentiment_start_score           235
cc_contractor_sentiment_end_score             234


In [144]:
print('Total rows with null values:', cc_calls.isnull().any(axis=1).sum())

Total rows with null values: 21236


In [145]:
null_rows = cc_calls[cc_calls.isnull().any(axis=1)]
non_null_rows = cc_calls[~cc_calls.isnull().any(axis=1)]
print('Dropped:', null_rows['direction'].value_counts(normalize=True).to_dict())
print('Kept:   ', non_null_rows['direction'].value_counts(normalize=True).to_dict())

Dropped: {'Outbound': 0.9515445469956677, 'Inbound': 0.048455453004332266}
Kept:    {'Inbound': 0.6072115384615384, 'Outbound': 0.39278846153846153}


In [146]:
cc_calls = cc_calls.dropna()
cc_calls.shape

(10400, 30)

In [147]:
print('Total rows with null values:', cc_calls.isnull().any(axis=1).sum())

Total rows with null values: 0


Saving cleaned dataset

In [148]:
import os
os.makedirs('../../data/cleaned', exist_ok=True)
cc_calls.to_csv('../../data/cleaned/cc_calls_cleaned.csv', index=False)
print('Saved! Shape:', cc_calls.shape)

Saved! Shape: (10400, 30)
